# Dataset Schemas

Purpose: show the structure of every dataset in one place, so a person can cross-reference columns and types across sources.

**This notebook prints structure only: each dataset's shape (rows, columns) and its column types. It prints no data rows and no statistics.**

Run it from top to bottom in the project environment (`uv sync`, then select the `.venv` kernel). It needs the full files in `data/raw/` and `data/external/` (see the Datasets table in `README.md`).

In [1]:
from pathlib import Path
# Used to build file paths that work on any operating system.

import geopandas as gpd
# Used to read the zipped shapefiles, which Polars cannot read.

import polars as pl
# Used to read the CSV files and get their schemas.

# Find the project root (the folder that holds pyproject.toml) by walking up from the
# current folder, so the paths work no matter where Jupyter was started.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not find pyproject.toml in this folder or any parent folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"

# Only the 2024 ACS tables are loaded.
ACS_YEAR = "2024"

OSM_LOCATIONS_PATH = RAW_DIR / "flock_osm" / "osm_flock_us_locations_2026-10-30.csv"
OSM_HISTORY_PATH = RAW_DIR / "flock_osm" / "flock_us_historical_2026-10-30.csv"
FINDING_FLOCK_CITY_PATH = RAW_DIR / "finding_flock" / "findingflock-cameras-by-city.csv"
FINDING_FLOCK_STATE_PATH = RAW_DIR / "finding_flock" / "findingflock-cameras-by-state.csv"
ACS_POPULATION_PATH = RAW_DIR / "census" / "population" / ACS_YEAR / f"ACSDT5Y{ACS_YEAR}.B01003-Data.csv"
ACS_INCOME_PATH = RAW_DIR / "census" / "median_household_income" / ACS_YEAR / f"ACSDT5Y{ACS_YEAR}.B19013-Data.csv"
ACS_POVERTY_PATH = RAW_DIR / "census" / "poverty" / ACS_YEAR / f"ACSST5Y{ACS_YEAR}.S1701-Data.csv"
ACS_RACE_PATH = RAW_DIR / "census" / "race_ethnicity" / ACS_YEAR / f"ACSDT5Y{ACS_YEAR}.B03002-Data.csv"
ELECTIONS_PATH = RAW_DIR / "elections" / "presidential" / "countypres_2000-2024.csv"
ATLAS_PATH = RAW_DIR / "atlas_of_surveillance" / "atlas_of_surveillance_2026-09-28.csv"
URBAN_RURAL_PATH = RAW_DIR / "census" / "urban_rural" / "2020" / "2020_UA_COUNTY.xlsx"
# One state's LODES file is the example (il = Illinois); every state's file has the same columns.
LODES_PATH = RAW_DIR / "lehd_lodes" / "wac" / "il_wac_S000_JT00_2023.csv.gz"
# One state's tract file is the example; 17 is the FIPS code for Illinois.
TRACTS_PATH = EXTERNAL_DIR / "census_boundaries" / "2024" / "tl_2024_17_tract.zip"
COUNTIES_PATH = EXTERNAL_DIR / "census_boundaries" / "counties" / "tl_2024_us_county.zip"
PRIMARY_ROADS_PATH = EXTERNAL_DIR / "census_boundaries" / "roads" / "2024" / "tl_2024_us_primaryroads.zip"
# One state's primary and secondary roads file is the example (17 = Illinois).
STATE_ROADS_PATH = EXTERNAL_DIR / "census_boundaries" / "roads" / "2024" / "tl_2024_17_prisecroads.zip"
# The HPMS download covers Illinois only, saved in two formats with the same rows.
HPMS_CSV_PATH = EXTERNAL_DIR / "hpms" / "2024" / "hpms_illinois_2024.csv"
HPMS_GEOJSON_PATH = EXTERNAL_DIR / "hpms" / "2024" / "hpms_illinois_2024.geojson"

ALL_PATHS = [
    OSM_LOCATIONS_PATH,
    OSM_HISTORY_PATH,
    FINDING_FLOCK_CITY_PATH,
    FINDING_FLOCK_STATE_PATH,
    ACS_POPULATION_PATH,
    ACS_INCOME_PATH,
    ACS_POVERTY_PATH,
    ACS_RACE_PATH,
    ELECTIONS_PATH,
    ATLAS_PATH,
    URBAN_RURAL_PATH,
    LODES_PATH,
    TRACTS_PATH,
    COUNTIES_PATH,
    PRIMARY_ROADS_PATH,
    STATE_ROADS_PATH,
    HPMS_CSV_PATH,
    HPMS_GEOJSON_PATH,
]

# Report missing inputs but keep going: each section below handles its own missing file.
missing_paths = [path for path in ALL_PATHS if not path.is_file()]
if missing_paths:
    print(f"Missing {len(missing_paths)} of {len(ALL_PATHS)} input files:")
    for path in missing_paths:
        print(f"  {path.relative_to(PROJECT_ROOT)}")
else:
    print(f"All {len(ALL_PATHS)} input files found.")


def print_structure(shape: tuple[int, int], column_types: dict) -> None:
    """Print a dataset's shape, then one line per column with its data type."""
    # One column per line is easier to cross-reference than the one-line schema printout.
    print(f"shape: {shape}")
    for column_name, column_type in column_types.items():
        print(f"  {column_name}: {column_type}")

All 18 input files found.


## OSM camera locations

`data/raw/flock_osm/osm_flock_us_locations_2026-10-30.csv`

**What it provides:** A snapshot of the Flock license plate reader cameras that volunteers have mapped in OpenStreetMap. Each row is one mapped camera with its location and all of its OSM tags.

In [2]:
# scan_csv builds a lazy query instead of loading the file. infer_schema_length=None
# makes Polars look at every row when guessing column types, so a type that only shows
# up late in the file is not missed. The row count is a lazy count, not a full load.
try:
    locations = pl.scan_csv(OSM_LOCATIONS_PATH, infer_schema_length=None)
    locations_schema = locations.collect_schema()
    locations_rows = locations.select(pl.len()).collect().item()
    print_structure((locations_rows, len(locations_schema)), locations_schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load OSM camera locations: {str(error).splitlines()[0]}")

shape: (112830, 20)
  osm_type: String
  osm_id: Int64
  lat: Float64
  lon: Float64
  timestamp: String
  version: Int64
  changeset: Int64
  user: String
  man_made: String
  surveillance_type: String
  manufacturer: String
  brand: String
  operator: String
  camera_direction: String
  direction: String
  camera_type: String
  camera_mount: String
  camera_angle: Int64
  surveillance_zone: String
  all_tags: String


## OSM mapping history

`data/raw/flock_osm/flock_us_historical_2026-10-30.csv`

**What it provides:** The OSM edit history of the same mapped Flock cameras. It gives mapping dates for each camera, such as when it was first added and first tagged as Flock, not when it was installed.

In [3]:
# Same lazy approach as the locations file.
try:
    history = pl.scan_csv(OSM_HISTORY_PATH, infer_schema_length=None)
    history_schema = history.collect_schema()
    history_rows = history.select(pl.len()).collect().item()
    print_structure((history_rows, len(history_schema)), history_schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load OSM mapping history: {str(error).splitlines()[0]}")

shape: (112527, 15)
  osm_id: Int64
  lat: Float64
  lon: Float64
  manufacturer: String
  operator: String
  version: Int64
  last_edit: String
  changeset: Int64
  user: String
  history_ok: Boolean
  first_insert_date: String
  first_insert_version: Int64
  first_tagged_date: String
  first_tagged_version: Int64
  first_tagged_changeset: Int64


## Finding Flock

`data/raw/finding_flock/findingflock-cameras-by-city.csv` and `data/raw/finding_flock/findingflock-cameras-by-state.csv`

**What it provides:** Camera counts by city and by state from findingflock.com, an independent tracker whose counts include other vendors as well as Flock. It serves as a check on the OSM map.

The city GeoJSON in the same folder is not included here.

In [4]:
# Two separate try blocks, so a problem with one file does not hide the other.
try:
    finding_flock_city = pl.read_csv(FINDING_FLOCK_CITY_PATH, infer_schema_length=None)
    print("By city:")
    print_structure(finding_flock_city.shape, finding_flock_city.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load Finding Flock (by city): {str(error).splitlines()[0]}")

try:
    finding_flock_state = pl.read_csv(FINDING_FLOCK_STATE_PATH, infer_schema_length=None)
    print("By state:")
    print_structure(finding_flock_state.shape, finding_flock_state.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load Finding Flock (by state): {str(error).splitlines()[0]}")

By city:
shape: (2422, 20)
  geoid: Int64
  city: String
  place_name: String
  state: String
  state_name: String
  cameras: Int64
  land_sq_mi: Float64
  cameras_per_sq_mi: Float64
  flock: Int64
  motorola: Int64
  genetec: Int64
  leonardo: Int64
  neology: Int64
  other_vendor: Int64
  unknown_vendor: Int64
  min_lon: Float64
  min_lat: Float64
  max_lon: Float64
  max_lat: Float64
  url: String
By state:
shape: (52, 13)
  state: String
  state_name: String
  cameras_statewide: Int64
  cameras_in_listed_cities: Int64
  listed_cities: Int64
  flock: Int64
  motorola: Int64
  genetec: Int64
  leonardo: Int64
  neology: Int64
  other_vendor: Int64
  unknown_vendor: Int64
  url: String


## ACS 5-year: population (B01003)

`data/raw/census/population/2024/ACSDT5Y2024.B01003-Data.csv`

**What it provides:** Census Bureau survey estimates for every census tract on population, median household income, poverty and race/ethnicity. Each table comes with column metadata and table notes.

In [5]:
# The Census download repeats the column labels in the row after the header
# (for example "Estimate!!Total"). skip_rows_after_header=1 skips that row so the
# column types come from the data, not from the label text.
try:
    acs_population = pl.read_csv(ACS_POPULATION_PATH, skip_rows_after_header=1, infer_schema_length=None)
    print(f"ACS year loaded: {ACS_YEAR}")
    print_structure(acs_population.shape, acs_population.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load ACS population: {str(error).splitlines()[0]}")

ACS year loaded: 2024
shape: (85382, 5)
  GEO_ID: String
  NAME: String
  B01003_001E: Int64
  B01003_001M: String
  : String


## ACS 5-year: median household income (B19013)

`data/raw/census/median_household_income/2024/ACSDT5Y2024.B19013-Data.csv`

**What it provides:** Census Bureau survey estimates for every census tract on population, median household income, poverty and race/ethnicity. Each table comes with column metadata and table notes.

In [6]:
# Skip the repeated label row, as in the population table.
try:
    acs_income = pl.read_csv(ACS_INCOME_PATH, skip_rows_after_header=1, infer_schema_length=None)
    print(f"ACS year loaded: {ACS_YEAR}")
    print_structure(acs_income.shape, acs_income.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load ACS median household income: {str(error).splitlines()[0]}")

ACS year loaded: 2024
shape: (85382, 5)
  GEO_ID: String
  NAME: String
  B19013_001E: String
  B19013_001M: String
  : String


## ACS 5-year: poverty (S1701)

`data/raw/census/poverty/2024/ACSST5Y2024.S1701-Data.csv`

**What it provides:** Census Bureau survey estimates for every census tract on population, median household income, poverty and race/ethnicity. Each table comes with column metadata and table notes.

In [7]:
# Skip the repeated label row, as in the population table.
try:
    acs_poverty = pl.read_csv(ACS_POVERTY_PATH, skip_rows_after_header=1, infer_schema_length=None)
    print(f"ACS year loaded: {ACS_YEAR}")
    print_structure(acs_poverty.shape, acs_poverty.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load ACS poverty: {str(error).splitlines()[0]}")

ACS year loaded: 2024
shape: (85382, 375)
  GEO_ID: String
  NAME: String
  S1701_C01_001E: Int64
  S1701_C01_001M: Int64
  S1701_C01_002E: Int64
  S1701_C01_002M: Int64
  S1701_C01_003E: Int64
  S1701_C01_003M: Int64
  S1701_C01_004E: Int64
  S1701_C01_004M: Int64
  S1701_C01_005E: Int64
  S1701_C01_005M: Int64
  S1701_C01_006E: Int64
  S1701_C01_006M: Int64
  S1701_C01_007E: Int64
  S1701_C01_007M: Int64
  S1701_C01_008E: Int64
  S1701_C01_008M: Int64
  S1701_C01_009E: Int64
  S1701_C01_009M: Int64
  S1701_C01_010E: Int64
  S1701_C01_010M: Int64
  S1701_C01_011E: Int64
  S1701_C01_011M: Int64
  S1701_C01_012E: Int64
  S1701_C01_012M: Int64
  S1701_C01_013E: Int64
  S1701_C01_013M: Int64
  S1701_C01_014E: Int64
  S1701_C01_014M: Int64
  S1701_C01_015E: Int64
  S1701_C01_015M: Int64
  S1701_C01_016E: Int64
  S1701_C01_016M: Int64
  S1701_C01_017E: Int64
  S1701_C01_017M: Int64
  S1701_C01_018E: Int64
  S1701_C01_018M: Int64
  S1701_C01_019E: Int64
  S1701_C01_019M: Int64
  S1701_C01_02

## ACS 5-year: race/ethnicity (B03002)

`data/raw/census/race_ethnicity/2024/ACSDT5Y2024.B03002-Data.csv`

**What it provides:** Census Bureau survey estimates for every census tract on population, median household income, poverty and race/ethnicity. Each table comes with column metadata and table notes.

In [8]:
# Skip the repeated label row, as in the population table.
try:
    acs_race = pl.read_csv(ACS_RACE_PATH, skip_rows_after_header=1, infer_schema_length=None)
    print(f"ACS year loaded: {ACS_YEAR}")
    print_structure(acs_race.shape, acs_race.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load ACS race/ethnicity: {str(error).splitlines()[0]}")

ACS year loaded: 2024
shape: (85382, 45)
  GEO_ID: String
  NAME: String
  B03002_001E: Int64
  B03002_001M: String
  B03002_002E: Int64
  B03002_002M: String
  B03002_003E: Int64
  B03002_003M: Int64
  B03002_004E: Int64
  B03002_004M: Int64
  B03002_005E: Int64
  B03002_005M: Int64
  B03002_006E: Int64
  B03002_006M: Int64
  B03002_007E: Int64
  B03002_007M: Int64
  B03002_008E: Int64
  B03002_008M: Int64
  B03002_009E: Int64
  B03002_009M: Int64
  B03002_010E: Int64
  B03002_010M: Int64
  B03002_011E: Int64
  B03002_011M: Int64
  B03002_012E: Int64
  B03002_012M: String
  B03002_013E: Int64
  B03002_013M: Int64
  B03002_014E: Int64
  B03002_014M: Int64
  B03002_015E: Int64
  B03002_015M: Int64
  B03002_016E: Int64
  B03002_016M: Int64
  B03002_017E: Int64
  B03002_017M: Int64
  B03002_018E: Int64
  B03002_018M: Int64
  B03002_019E: Int64
  B03002_019M: Int64
  B03002_020E: Int64
  B03002_020M: Int64
  B03002_021E: Int64
  B03002_021M: Int64
  : String


## County presidential returns

`data/raw/elections/presidential/countypres_2000-2024.csv`

**What it provides:** Vote counts by county and candidate for every presidential election from 2000 to 2024, compiled by the MIT Election Data and Science Lab.

In [9]:
try:
    elections = pl.read_csv(ELECTIONS_PATH, infer_schema_length=None)
    print_structure(elections.shape, elections.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load county presidential returns: {str(error).splitlines()[0]}")

shape: (94151, 12)
  state: String
  county_name: String
  year: Int64
  state_po: String
  county_fips: String
  office: String
  candidate: String
  party: String
  candidatevotes: String
  totalvotes: Int64
  version: Int64
  mode: String


## Atlas of Surveillance

`data/raw/atlas_of_surveillance/atlas_of_surveillance_2026-09-28.csv`

**What it provides:** EFF's database of which U.S. law enforcement agencies use which surveillance technologies, with one row per agency and technology. It is used for exploration only.

In [10]:
try:
    atlas = pl.read_csv(ATLAS_PATH, infer_schema_length=None)
    print_structure(atlas.shape, atlas.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load Atlas of Surveillance: {str(error).splitlines()[0]}")

shape: (15135, 28)
  AOSNUMBER: String
  NEWAOSNUMBER (ORI9): String
  City: String
  County: String
  State: String
  Agency: String
  Type of LEA: String
  Summary: String
  Type of Juris: String
  Technology: String
  TECH ABV: String
  Vendor: String
  Link 1: String
  Link 1 Snapshot: String
  Link 1 Source: String
  Link 1 Type: String
  Link 1 Date: String
  Link 2: String
  Link 2 Snapshot: String
  Link 2 Source: String
  Link 2 Type: String
  Link 2 Date: String
  Link 3: String
  Link 3 Snapshot: String
  Link 3 Source: String
  Link 3 Type: String
  Link 3 Date: String
  Other Links: String


## Census urban/rural (2020)

`data/raw/census/urban_rural/2020/2020_UA_COUNTY.xlsx`

**What it provides:** A Census Bureau table that splits each county's population, housing and land area into urban and rural parts, based on the 2020 Census urban areas.

The workbook has three sheets: `2020_UA_COUNTY` (the county table), `CT_2022` (Connecticut's 2022 planning regions, which replaced its counties) and `FieldDescriptions_Notes` (text notes, not read here).

In [11]:
# Polars reads Excel through the fastexcel package. Each sheet is named explicitly,
# because by default read_excel only reads the first sheet.
try:
    urban_rural = pl.read_excel(URBAN_RURAL_PATH, sheet_name="2020_UA_COUNTY")
    print("Sheet 2020_UA_COUNTY:")
    print_structure(urban_rural.shape, urban_rural.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load Census urban/rural (2020_UA_COUNTY): {str(error).splitlines()[0]}")

try:
    urban_rural_connecticut = pl.read_excel(URBAN_RURAL_PATH, sheet_name="CT_2022")
    print("Sheet CT_2022:")
    print_structure(urban_rural_connecticut.shape, urban_rural_connecticut.schema)
except Exception as error:
    print(f"Could not load Census urban/rural (CT_2022): {str(error).splitlines()[0]}")

Sheet 2020_UA_COUNTY:
shape: (3234, 31)
  STATE: String
  COUNTY: String
  STATE_NAME: String
  COUNTY_NAME: String
  POP_COU: Int64
  HOU_COU: Int64
  ALAND_COU: Int64
  ALAND_Mi²_COU: Float64
  POPDEN_COU: Float64
  HOUDEN_COU: Float64
  TOTALBLOCKS: Int64
  POP_URB: Int64
  POPPCT_URB: Float64
  HOU_URB: Int64
  HOUPCT_URB: Float64
  ALAND_URB: Int64
  ALAND_Mi²_URB: Float64
  ALAND_PCT_URB: Float64
  POPDEN_URB: Float64
  HOUDEN_URB: Float64
  URBANBLOCKS: Int64
  POP_RUR: Int64
  POPPCT_RUR: Float64
  HOU_RUR: Int64
  HOUPCT_RUR: Float64
  ALAND_RUR: Int64
  ALAND_Mi²_RUR: Float64
  ALAND_PCT_RUR: Float64
  POPDEN_RUR: Float64
  HOUDEN_RUR: Float64
  RURALBLOCKS: Int64
Sheet CT_2022:
shape: (9, 31)
  STATE: String
  COUNTY: Int64
  STATE_NAME: String
  COUNTY_NAME: String
  POP_COU: Int64
  HOU_COU: Int64
  ALAND_COU: Int64
  ALAND_Mi²_COU: Float64
  POPDEN_COU: Float64
  HOUDEN_COU: Float64
  TOTALBLOCKS: Int64
  POP_URB: Int64
  POPPCT_URB: Float64
  HOU_URB: Int64
  HOUPCT_URB:

## LEHD LODES workplace jobs (Illinois example)

`data/raw/lehd_lodes/wac/il_wac_S000_JT00_2023.csv.gz`

**What it provides:** Census Bureau counts of jobs by the census block where people work, one file per state. They show where people work during the day, not where they live.

Every state's file has the same columns. Most states are 2023; Alaska is 2016 and Michigan is 2021.

In [12]:
# read_csv opens the gzipped file directly. Polars guesses w_geocode (the 15-digit census
# block code) as an integer, which would drop the leading zero for states with a FIPS code
# below 10; read it with schema_overrides={"w_geocode": pl.String} when joining.
try:
    lodes = pl.read_csv(LODES_PATH, infer_schema_length=None)
    print_structure(lodes.shape, lodes.schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load LEHD LODES (Illinois): {str(error).splitlines()[0]}")

shape: (92112, 53)
  w_geocode: Int64
  C000: Int64
  CA01: Int64
  CA02: Int64
  CA03: Int64
  CE01: Int64
  CE02: Int64
  CE03: Int64
  CNS01: Int64
  CNS02: Int64
  CNS03: Int64
  CNS04: Int64
  CNS05: Int64
  CNS06: Int64
  CNS07: Int64
  CNS08: Int64
  CNS09: Int64
  CNS10: Int64
  CNS11: Int64
  CNS12: Int64
  CNS13: Int64
  CNS14: Int64
  CNS15: Int64
  CNS16: Int64
  CNS17: Int64
  CNS18: Int64
  CNS19: Int64
  CNS20: Int64
  CR01: Int64
  CR02: Int64
  CR03: Int64
  CR04: Int64
  CR05: Int64
  CR07: Int64
  CT01: Int64
  CT02: Int64
  CD01: Int64
  CD02: Int64
  CD03: Int64
  CD04: Int64
  CS01: Int64
  CS02: Int64
  CFA01: Int64
  CFA02: Int64
  CFA03: Int64
  CFA04: Int64
  CFA05: Int64
  CFS01: Int64
  CFS02: Int64
  CFS03: Int64
  CFS04: Int64
  CFS05: Int64
  createdate: Int64


## TIGER/Line tracts (2024, Illinois example) — read with GeoPandas

`data/external/census_boundaries/2024/tl_2024_17_tract.zip`

**What it provides:** Census Bureau shapefiles of census tract boundaries, one file per state. They give each tract's shape and ID, which link locations to the ACS tract estimates.

Every state's tract file has the same columns, so one state stands in for all of them.

In [13]:
# GeoPandas, not Polars: Polars cannot read shapefiles. GeoPandas reads the zipped
# shapefile directly. dtypes are the pandas column types; "geometry" holds the shapes.
try:
    tracts = gpd.read_file(TRACTS_PATH)
    print_structure(tracts.shape, dict(tracts.dtypes))
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load TIGER/Line tracts: {str(error).splitlines()[0]}")

shape: (3265, 14)
  STATEFP: str
  COUNTYFP: str
  TRACTCE: str
  GEOID: str
  GEOIDFQ: str
  NAME: str
  NAMELSAD: str
  MTFCC: str
  FUNCSTAT: str
  ALAND: int64
  AWATER: int64
  INTPTLAT: str
  INTPTLON: str
  geometry: geometry


## TIGER/Line counties (2024) — read with GeoPandas

`data/external/census_boundaries/counties/tl_2024_us_county.zip`

**What it provides:** A national Census Bureau shapefile of county boundaries, with each county's shape and ID.

In [14]:
# GeoPandas, not Polars (see the tracts cell). The full national file is read.
try:
    counties = gpd.read_file(COUNTIES_PATH)
    print_structure(counties.shape, dict(counties.dtypes))
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load TIGER/Line counties: {str(error).splitlines()[0]}")

shape: (3235, 19)
  STATEFP: str
  COUNTYFP: str
  COUNTYNS: str
  GEOID: str
  GEOIDFQ: str
  NAME: str
  NAMELSAD: str
  LSAD: str
  CLASSFP: str
  MTFCC: str
  CSAFP: str
  CBSAFP: str
  METDIVFP: str
  FUNCSTAT: str
  ALAND: int64
  AWATER: int64
  INTPTLAT: str
  INTPTLON: str
  geometry: geometry


## TIGER/Line national primary roads (2024) — read with GeoPandas

`data/external/census_boundaries/roads/2024/tl_2024_us_primaryroads.zip`

**What it provides:** Census Bureau shapefiles of the primary and secondary roads in each state, plus a national file of primary roads. They show where the major roads are, with each road's name and type.

The per-state `tl_2024_<FIPS>_prisecroads.zip` files are in the next section (Illinois example).

In [15]:
# GeoPandas, not Polars (see the tracts cell). The full national file is read.
try:
    primary_roads = gpd.read_file(PRIMARY_ROADS_PATH)
    print_structure(primary_roads.shape, dict(primary_roads.dtypes))
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load TIGER/Line primary roads: {str(error).splitlines()[0]}")

shape: (17522, 5)
  LINEARID: str
  FULLNAME: str
  RTTYP: str
  MTFCC: str
  geometry: geometry


## TIGER/Line primary and secondary roads (2024, Illinois example) — read with GeoPandas

`data/external/census_boundaries/roads/2024/tl_2024_17_prisecroads.zip`

**What it provides:** Census Bureau shapefiles of the primary and secondary roads in each state, plus a national file of primary roads. They show where the major roads are, with each road's name and type.

Every state's file has the same columns, so one state stands in for all of them.

In [16]:
# GeoPandas, not Polars (see the tracts cell).
try:
    state_roads = gpd.read_file(STATE_ROADS_PATH)
    print_structure(state_roads.shape, dict(state_roads.dtypes))
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load TIGER/Line primary and secondary roads (Illinois): {str(error).splitlines()[0]}")

shape: (9036, 5)
  LINEARID: str
  FULLNAME: str
  RTTYP: str
  MTFCC: str
  geometry: geometry


## HPMS road sections (2024, Illinois only), CSV

`data/external/hpms/2024/hpms_illinois_2024.csv`

**What it provides:** Federal Highway Administration records of Illinois road sections, each with its road class, length and, where measured, its average annual daily traffic (AADT).

The road geometry is stored as WKT text in the `line` column.

In [17]:
# Same lazy approach as the OSM files.
try:
    hpms = pl.scan_csv(HPMS_CSV_PATH, infer_schema_length=None)
    hpms_schema = hpms.collect_schema()
    hpms_rows = hpms.select(pl.len()).collect().item()
    print_structure((hpms_rows, len(hpms_schema)), hpms_schema)
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load HPMS (CSV): {str(error).splitlines()[0]}")

shape: (445757, 8)
  stateid: Int64
  county_id: Int64
  urban_id: Int64
  f_system: Int64
  aadt: Int64
  sectionlength: Float64
  route_id: String
  line: String


## HPMS road sections (2024, Illinois only), GeoJSON — read with GeoPandas

`data/external/hpms/2024/hpms_illinois_2024.geojson`

**What it provides:** Federal Highway Administration records of Illinois road sections, each with its road class, length and, where measured, its average annual daily traffic (AADT).

Same rows as the CSV, with the geometry as real shapes.

In [18]:
# GeoPandas, not Polars (see the tracts cell). The full file is read.
try:
    hpms_shapes = gpd.read_file(HPMS_GEOJSON_PATH)
    print_structure(hpms_shapes.shape, dict(hpms_shapes.dtypes))
except Exception as error:
    # Keep going: print the first line of the error and move to the next dataset.
    print(f"Could not load HPMS (GeoJSON): {str(error).splitlines()[0]}")

shape: (445757, 8)
  f_system: str
  sectionlength: str
  urban_id: str
  route_id: str
  aadt: str
  stateid: str
  county_id: str
  geometry: geometry
